Ten skrypt tworzy system wieloagentowy wykorzystujący CrewAI do analizowania bazy danych wynagrodzeń poprzez sekwencyjną współpracę trzech agentów: programisty SQL, analityka danych i twórcy raportów. System łączy się z bazą SQLite zawierającą dane o zarobkach, definiuje narzędzia do operacji na bazie (lista tabel, schematy, wykonywanie zapytań SQL) i wykonuje analizę wpływu lokalizacji, rozmiaru firmy i doświadczenia pracownika na wysokość wynagrodzenia.

# Setup

In [ ]:
!uv pip install langchain_community crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.8/170.8 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.7/169.7 kB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 15.

*   `langchain_community`: Biblioteka ułatwiająca tworzenie aplikacji wykorzystujących modele językowe (LLM).
*   `crewai`: Framework do budowania agentów AI współpracujących ze sobą w celu realizacji złożonych zadań.
*   `crewai_tools`: Zestaw narzędzi rozszerzających możliwości `crewai`.

In [53]:
# Standard library imports
import sqlite3
from textwrap import dedent

# Third-party imports
import pandas as pd
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
)
from langchain_community.tools import QuerySQLDatabaseTool
from langchain_community.utilities.sql_database import SQLDatabase

In [54]:
class CFG:
    model = "llama3.2"
    temp = 0.1

In [55]:
llm = LLM(model=f"ollama/{CFG.model}", base_url="http://127.0.0.1:11434")

# Dane

Źródło: https://www.kaggle.com/datasets/arnabchaki/data-science-salaries-2023

In [56]:
df = pd.read_csv("./content/ds_salaries.csv")
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


In [59]:
connection = sqlite3.connect("salaries.db")
df.to_sql(name="salaries", con=connection, if_exists="replace")

3755

Ten kod tworzy bazę danych SQLite i zapisuje zawartość ramki danych `pandas` do tabeli w tej bazie danych.

*   `connection = sqlite3.connect("salaries.db")`: Ta linia kodu nawiązuje połączenie z bazą danych SQLite o nazwie "salaries.db". Jeśli plik bazy danych nie istnieje, zostanie utworzony. Zmienna `connection` przechowuje obiekt reprezentujący to połączenie.
*   `df.to_sql(name="salaries", con=connection)`: Ta linia kodu wykorzystuje metodę `to_sql()` ramki danych `pandas` (`df`) do zapisania jej zawartości do bazy danych SQLite.
    *   `name="salaries"`: Określa nazwę tabeli w bazie danych, która zostanie utworzona lub do której zostaną dodane dane. W tym przypadku tabela będzie nazywać się "salaries".
    *   `con=connection`: Przekazuje obiekt połączenia z bazą danych (`connection`) jako argument, wskazując gdzie mają być zapisane dane.


In [60]:
db = SQLDatabase.from_uri("sqlite:///salaries.db")

Ten kod nawiązuje połączenie z bazą danych SQLite za pomocą klasy `SQLDatabase` z biblioteki `langchain_community.utilities.sql_database`.

*   `db = SQLDatabase.from_uri("sqlite:///salaries.db")`: Ta linia kodu tworzy instancję klasy `SQLDatabase`, która reprezentuje połączenie z bazą danych SQLite.
    *   `SQLDatabase.from_uri(...)`: To statyczna metoda klasy `SQLDatabase`, która przyjmuje URI (Uniform Resource Identifier) bazy danych jako argument.
    *   `"sqlite:///salaries.db"`:  To URI wskazujące na lokalny plik bazy danych SQLite o nazwie "salaries.db". Prefiks `"sqlite:///"` informuje bibliotekę, że ma połączyć się z bazą danych SQLite.

# Agentura


## Narzędzia

In [62]:
@tool("list_tables")
def list_tables() -> str:
    """List the available tables in the database"""
    return ListSQLDatabaseTool(db=db).invoke("")


list_tables.run()

Using Tool: list_tables


'salaries'

Ten kod definiuje narzędzie (tool) dla agentów AI, które pozwala na wyświetlenie listy tabel w bazie danych SQLite i następnie uruchamia to narzędzie.

*   `@tool("list_tables")`: To dekorator z biblioteki `crewai.tools`, który rejestruje funkcję `list_tables()` jako narzędzie dostępne dla agentów AI. Argument `"list_tables"` jest nazwą, pod jaką narzędzie będzie identyfikowane w systemie.

In [63]:
@tool("tables_schema")
def tables_schema(tables: str) -> str:
    """
    Input is a comma-separated list of tables, output is the schema and sample rows
    for those tables. Be sure that the tables actually exist by calling `list_tables` first!
    Example Input: table1, table2, table3
    """
    tool = InfoSQLDatabaseTool(db=db)
    return tool.invoke(tables)


print(tables_schema.run("salaries"))

Using Tool: tables_schema

CREATE TABLE salaries (
	"index" INTEGER, 
	work_year INTEGER, 
	experience_level TEXT, 
	employment_type TEXT, 
	job_title TEXT, 
	salary INTEGER, 
	salary_currency TEXT, 
	salary_in_usd INTEGER, 
	employee_residence TEXT, 
	remote_ratio INTEGER, 
	company_location TEXT, 
	company_size TEXT
)

/*
3 rows from salaries table:
index	work_year	experience_level	employment_type	job_title	salary	salary_currency	salary_in_usd	employee_residence	remote_ratio	company_location	company_size
0	2023	SE	FT	Principal Data Scientist	80000	EUR	85847	ES	100	ES	L
1	2023	MI	CT	ML Engineer	30000	USD	30000	US	100	US	S
2	2023	MI	CT	ML Engineer	25500	USD	25500	US	100	US	S
*/


Ten kod definiuje i używa narzędzie do analizy struktury tabel:

1. Dekorator `@tool("tables_schema")` rejestruje funkcję jako narzędzie o nazwie "tables_schema"

2. Funkcja `tables_schema()`:
   - przyjmuje parametr `tables` jako string z nazwami tabel oddzielonymi przecinkami
   - zwraca string z informacjami o strukturze tabel
   - używa `InfoSQLDatabaseTool` z langchain do pobrania informacji o tabelach
   - parametr db=db przekazuje połączenie do bazy

3. Dokumentacja funkcji w potrójnych cudzysłowach wyjaśnia:
   - format wejścia (nazwy tabel oddzielone przecinkami)
   - typ wyniku (schemat i przykładowe wiersze)
   - zalecenie sprawdzenia istnienia tabel przez `list_tables`
   - przykład wejścia: "table1, table2, table3"

4. `print(tables_schema.run("salaries"))` wyświetla strukturę tabeli "salaries", pokazując:
   - nazwy kolumn
   - typy danych
   - przykładowe wiersze z tabeli

In [64]:
@tool("execute_sql")
def execute_sql(sql_query: str) -> str:
    """Execute a SQL query against the database. Returns the result"""
    return QuerySQLDatabaseTool(db=db).invoke(sql_query)


execute_sql.run("SELECT * FROM salaries WHERE salary > 10000 LIMIT 5")

Using Tool: execute_sql


"[(0, 2023, 'SE', 'FT', 'Principal Data Scientist', 80000, 'EUR', 85847, 'ES', 100, 'ES', 'L'), (1, 2023, 'MI', 'CT', 'ML Engineer', 30000, 'USD', 30000, 'US', 100, 'US', 'S'), (2, 2023, 'MI', 'CT', 'ML Engineer', 25500, 'USD', 25500, 'US', 100, 'US', 'S'), (3, 2023, 'SE', 'FT', 'Data Scientist', 175000, 'USD', 175000, 'CA', 100, 'CA', 'M'), (4, 2023, 'SE', 'FT', 'Data Scientist', 120000, 'USD', 120000, 'CA', 100, 'CA', 'M')]"

In [65]:
@tool("check_sql")
def check_sql(sql_query: str) -> str:
    """
    Use this tool to double check if your query is correct before executing it. Always use this
    tool before executing a query with `execute_sql`.
    """
    return QuerySQLCheckerTool(db=db, llm=llm).invoke({"query": sql_query})


check_sql.run("SELECT * WHERE salary > 10000 LIMIT 5 table = salaries")

Using Tool: check_sql


ValidationError: 2 validation errors for QuerySQLCheckerTool
llm.is-instance[Runnable]
  Input should be an instance of Runnable [type=is_instance_of, input_value=<crewai.llm.LLM object at 0x161a1bc40>, input_type=LLM]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of
llm.is-instance[Runnable]
  Input should be an instance of Runnable [type=is_instance_of, input_value=<crewai.llm.LLM object at 0x161a1bc40>, input_type=LLM]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of

## Agenci

In [66]:
sql_dev = Agent(
    role="Senior Database Developer",
    goal="Construct and execute SQL queries based on a request",
    backstory=dedent(
        """
        You are an experienced database engineer who is master at creating efficient and complex SQL queries.
        You have a deep understanding of how different databases work and how to optimize queries.
        Use the `list_tables` to find available tables.
        Use the `tables_schema` to understand the metadata for the tables.
        Use the `execute_sql` to check your queries for correctness.
        Use the `check_sql` to execute queries against the database.
    """
    ),
    llm=llm,
    tools=[list_tables, tables_schema, execute_sql, check_sql],
    allow_delegation=False,
)

In [44]:
data_analyst = Agent(
    role="Senior Data Analyst",
    goal="You receive data from the database developer and analyze it",
    backstory=dedent(
        """
        You have deep experience with analyzing datasets using Python.
        Your work is always based on the provided data and is clear,
        easy-to-understand and to the point. You have attention
        to detail and always produce very detailed work (as long as you need).
    """
    ),
    llm=llm,
    allow_delegation=False,
)

In [67]:
report_writer = Agent(
    role="Senior Report Editor",
    goal="Write an executive summary type of report based on the work of the analyst",
    backstory=dedent(
        """
        Your writing still is well known for clear and effective communication.
        You always summarize long texts into bullet points that contain the most
        important details.
        """
    ),
    llm=llm,
    allow_delegation=False,
)

## Zadania

In [68]:
extract_data = Task(
    description="Extract data that is required for the query {query}.",
    expected_output="Database result for the query",
    agent=sql_dev,
)

In [69]:
analyze_data = Task(
    description="Analyze the data from the database and write an analysis for {query}.",
    expected_output="Detailed analysis text",
    agent=data_analyst,
    context=[extract_data],
)

In [70]:
write_report = Task(
    description=dedent(
        """
        Write an executive summary of the report from the analysis. The report
        must be less than 100 words.
    """
    ),
    expected_output="Markdown report",
    agent=report_writer,
    context=[analyze_data],
)

## załoga

In [71]:
crew = Crew(
    agents=[sql_dev, data_analyst, report_writer],
    tasks=[extract_data, analyze_data, write_report],
    process=Process.sequential,
    verbose=True,
    memory=False,
    output_log_file="crew.log",
)

# Test

In [72]:
inputs = {
    "query": "How does company location, size and employee experience affect the salary?"
}

result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 16a81092-836a-498a-b319-55255364dc23                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 08de0204-a5ed-413c-9b39-37a3a2447a22
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 08de0204-a5ed-413c-9b39-37a3a2447a22
       Status: Executing Task...
    └── 🤖 Agent: Senior Database Developer
            Status: In Progress

# Agent: Senior Database Developer
## Task: Extract data that is required for the query How does company location, size and employee experience affect the salary?.


🤖 Agent: Senior Database Developer
    Status: In Progress

...

 Error during LLM call: litellm.APIConnectionError: list index out of range
Traceback (most recent call last):
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/main.py", line 2904, in completion
    response = base_llm_http_handler.completion(
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 270, in completion
    data = provider_config.transform_request(
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/llms/ollama/completion/transformation.py", line 322, in transform_request
    modified_prompt = ollama_pt(model=model, messages=messages)
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/prompt_templates/f

🚀 Crew: crew
└── 📋 Task: 08de0204-a5ed-413c-9b39-37a3a2447a22
       Assigned to: Senior Database Developer
       Status: ❌ Failed
    └── 🤖 Agent: Senior Database Developer
            Status: In Progress
        └── ❌ LLM Failed

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 08de0204-a5ed-413c-9b39-37a3a2447a22                                                                     │
│  Agent: Senior Database Developer                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 16a81092-836a-498a-b319-55255364dc23                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

APIConnectionError: litellm.APIConnectionError: list index out of range
Traceback (most recent call last):
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/main.py", line 2904, in completion
    response = base_llm_http_handler.completion(
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 270, in completion
    data = provider_config.transform_request(
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/llms/ollama/completion/transformation.py", line 322, in transform_request
    modified_prompt = ollama_pt(model=model, messages=messages)
  File "/Users/marcinpilarczyk/projects/python/AI-Notes/_personal/course/agents/day_05/.venv/lib/python3.10/site-packages/litellm/litellm_core_utils/prompt_templates/facto

In [51]:
!cat crew.log.txt

2025-05-14 10:48:47: task_name="None", task="Extract data that is required for the query How does company location, size and employee experience affect the salary?.", agent="Senior Database Developer", status="started"
2025-05-14 10:51:35: task_name="None", task="Extract data that is required for the query How does company location, size and employee experience affect the salary?.", agent="Senior Database Developer", status="started"
2025-05-14 10:53:57: task_name="None", task="Extract data that is required for the query How does company location, size and employee experience affect the salary?.", agent="Senior Database Developer", status="started"
